Setting Up the Environment

In [ ]:
!pip install fastapi uvicorn pyngrok youtube-transcript-api "transformers<5" torch streamlit -q

Initialize BART Model and Start Background FastAPI Server

In [ ]:
import uvicorn
import threading
import time
import socket
from fastapi import FastAPI, Request, HTTPException
from pyngrok import ngrok, conf
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline
from urllib.parse import urlparse, parse_qs

NGROK_TOKEN = "ADD YOURS" 
API_KEY = "secret123" 

app = FastAPI()

print("Loading summarization model...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

def extract_video_id(url: str) -> str:
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)
    video_ids = qs.get('v')
    if not video_ids:
        if 'youtu.be' in parsed.netloc: return parsed.path.lstrip('/')
        raise ValueError(f"No video id found in URL: {url}")
    return video_ids[0]

@app.post("/summarize")
async def summarize_video(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    
    data = await req.json()
    url = data.get("url")
    
    try:
        video_id = extract_video_id(url)
        
        api = YouTubeTranscriptApi()
        fetched = api.fetch(video_id, languages=['en'])
        
        
        text = "\n".join(snippet.text for snippet in fetched)
        
        summary = summarizer(text[:3000], max_length=130, min_length=30, do_sample=False)
        
        return {
            "video_id": video_id,
            "summary": summary[0]['summary_text']
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

api_port = free_port()

def run(): 
    uvicorn.run(app, host="127.0.0.1", port=api_port)

threading.Thread(target=run, daemon=True).start()
time.sleep(3)
print(f"FastAPI is running on port: {api_port}")

Streamlit

In [ ]:
streamlit_code = f"""
import streamlit as st
import requests

st.set_page_config(page_title="YouTube Summarizer", page_icon="📺")
st.title("📺 YouTube Video Summarizer")

API_URL = "http://127.0.0.1:{api_port}/summarize"
API_KEY = "secret123"

url = st.text_input("Enter YouTube URL:")

if st.button("Summarize"):
    if url:
        with st.spinner("Processing..."):
            try:
                headers = {{"Authorization": f"Bearer {{API_KEY}}"}}
                resp = requests.post(API_URL, json={{"url": url}}, headers=headers)
                if resp.status_code == 200:
                    st.success("Done!")
                    st.subheader("Summary")
                    st.write(resp.json()['summary'])
                else:
                    st.error(f"Error: {{resp.text}}")
            except Exception as e:
                st.error(f"Connection failed: {{e}}")
"""

with open("streamlit_app.py", "w") as f:
    f.write(streamlit_code)

Launch via ngrok Tunnel

In [ ]:
from pyngrok import ngrok, conf
import subprocess

conf.get_default().auth_token = NGROK_TOKEN

try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        if ":8501" in t.config['addr']:
            ngrok.disconnect(t.public_url)
            
    public_url = ngrok.connect(8501).public_url
    print(f"Click here to open the Streamlit UI: {public_url}")
except Exception as e:
    print(f"ngrok error: {e}")

subprocess.Popen(["streamlit", "run", "streamlit_app.py", "--server.port", "8501"])